Metric that can be used to evaluate the performance of the LLM in identifying and calling the required tools to complete a given task. This metric needs `user_input` and `reference_tool_calls` to evaluate the performance of the LLM in identifying and calling the required tools to complete a given task. The metric is computed by comparing the `reference_tool_calls` with the Tool calls made by the AI. The values range between 0 and 1, with higher values indicating better performance.

In [ ]:
from agents.calendar.worker import google_calendar_worker
from dotenv import load_dotenv
import os
from IPython.display import Markdown
from langchain_core.messages import HumanMessage


import sys
from pathlib import Path

project_root = Path(".").resolve().parent.parent.parent
sys.path.append(str(project_root))
os.chdir(project_root)


load_dotenv()


test_user_email = "manuel.quesada@uhasselt.be"
user_id = "manuel-test"
config = {"configurable": {"thread_id": "testing_state_graph01"}}

In [2]:
response = await google_calendar_worker.ainvoke(
    {"workers_messages": [HumanMessage(content="Create an event or one hour for 25th of August at any free time to carry my documents to the bank")], "user_id": user_id,
     "account_name": test_user_email
     }, config)
response["workers_messages"]

[SystemMessage(content='## ROLE:\nYou excel at managing calendars and providing necessary information using the available tools.\n\n## Your goal is to achieve the following outcome:\nManage calendar operations effectively while ensuring data accuracy and providing clear responses.\n\n## Your task is as follows:\nAlways use this information as the foundation for managing the calendar.\n\n### **Time Formats**\n1. Always use dates in ISO format with timezone (e.g., `"2025-12-06T06:00:00+01:00"` or `"2025-05-07T00:00:00+02:00"`).\n2. By default, always use the **Europe/Paris** timezone unless specified otherwise by the user.\n\n### **Event Management Rules**\n\n#### **Creating Events**\n- Always use the tools provided to create new events.\n- The user must provide at least the start date for the event. If the user\'s request only includes the duration, set the start date to the current date and use the provided duration. If the user does not provide a duration, set it according to the even

In [3]:
from ragas.integrations.langgraph import convert_to_ragas_messages

ragas_trace = convert_to_ragas_messages(response["workers_messages"])
ragas_trace

[HumanMessage(content='Create an event or one hour for 25th of August at any free time to carry my documents to the bank', metadata=None, type='human'),
 AIMessage(content='', metadata=None, type='ai', tool_calls=[ToolCall(name='create_event', args={'start_datetime': '2025-08-25T00:00:00+02:00', 'title': 'Carry Documents to the Bank', 'event_duration_hour': 1, 'ignore_conflicts': False})]),
 ToolMessage(content="{'details': 'Event created successfully.', 'events': [{'title': 'Carry Documents to the Bank', 'start': '2025-08-25T00:00:00+02:00', 'end': '2025-08-25T01:30:00+02:00', 'event_link': 'https://www.google.com/calendar/event?eid=bDNobHY2MWplM28yaHJxYnFxOGdzdW1xOWcgbWFudWVsLnF1ZXNhZGFAdWhhc3NlbHQuYmU', 'attendees': [], 'meet_link': 'https://meet.google.com/nwt-yvdv-eii', 'description': None}]}", metadata=None, type='tool'),
 AIMessage(content='```json\n{\n  "answer": "The event has been successfully created based on the account: manuel.quesada@uhasselt.be.",\n  "events": [\n    {\n

In [ ]:
from ragas.metrics import ToolCallAccuracy
from ragas.dataset_schema import MultiTurnSample
from ragas.messages import HumanMessage, AIMessage, ToolMessage, ToolCall

from ragas.metrics._string import NonLLMStringSimilarity
from ragas.metrics._tool_call_accuracy import ToolCallAccuracy


sample = MultiTurnSample(
    user_input=ragas_trace,
    reference_tool_calls=[
        # ToolCall(name="find_free_slots", args={'start_date': '2025-08-25', 'end_date': '2025-08-25', 'min_duration': '1'}
        #          ),
        ToolCall(name="create_event", args={
                 'start_datetime': '2025-08-25', 'event_duration_hour': 1, 'ignore_conflicts': False})
    ],
    rubrics={"success": "The tool call was successful and the output is correct.",
             "helpfulness": "The tool call was helpful and the output is correct."}
    # reference_topics=["Event creation in Google Calendar"]
)

metric = ToolCallAccuracy()
metric.arg_comparison_metric = NonLLMStringSimilarity()
response = await metric.multi_turn_ascore(sample)
response

0.6666666666666666

## Agent Goal accuracy

Agent goal accuracy is a metric that can be used to evaluate the performance of the LLM in identifying and achieving the goals of the user. This is a binary metric, with 1 indicating that the AI has achieved the goal and 0 indicating that the AI has not achieved the goal.

In [21]:
from ragas.dataset_schema import MultiTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.messages import HumanMessage, AIMessage, ToolMessage, ToolCall
from ragas.metrics import AgentGoalAccuracyWithReference, AgentGoalAccuracyWithoutReference, MetricOutputType

from langchain_openai import ChatOpenAI

load_dotenv()

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))

sample = MultiTurnSample(user_input=ragas_trace,
                         #  reference="Event creation in Google Calendar for 2025-08-25 and all details in the AI response"
                         )

# scorer = AgentGoalAccuracyWithReference(
#     llm=evaluator_llm, output_type=MetricOutputType.RANKING)
# await scorer.multi_turn_ascore(sample)

scorer = AgentGoalAccuracyWithoutReference(llm=evaluator_llm)
await scorer.multi_turn_ascore(sample)

0.0

In [22]:
sample = MultiTurnSample(
    user_input=ragas_trace,
    reference="Event creation in Google Calendar for 2025-08-25 and all details in the AI response",
)

scorer = AgentGoalAccuracyWithReference()

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
scorer.llm = evaluator_llm
await scorer.multi_turn_ascore(sample)

0.0